# 04. Model Clustering IDM 2024
Notebook ini memuat persiapan fitur, pencarian jumlah klaster optimal, training K-Means, visualisasi klaster, dan penyimpanan hasil.

Notebook ini membaca `data/idm_2024_clean.csv` yang dihasilkan dari notebook 03_EDA.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='muted')

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists() and (BASE_DIR.parent / 'data').exists():
    BASE_DIR = BASE_DIR.parent

CLEAN_DATA_PATH = BASE_DIR / 'data' / 'idm_2024_clean.csv'
CLUSTERED_DATA_PATH = BASE_DIR / 'data' / 'idm_2024_clustered.csv'

print(f'Clean data    : {CLEAN_DATA_PATH}')
print(f'Clustered data: {CLUSTERED_DATA_PATH}')

In [ ]:
df = pd.read_csv(CLEAN_DATA_PATH)
feature_cols = ['IKS_2024', 'IKE_2024', 'IKL_2024']
df_cluster = df[feature_cols].dropna().copy()
print(f'Cluster dataset: {len(df_cluster):,} rows x {df_cluster.shape[1]} columns')
display(df_cluster.head(3))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print('Statistik setelah standardisasi:')
display(pd.DataFrame(X_scaled, columns=feature_cols).describe().round(4))

In [ ]:
np.random.seed(42)
sample_size = min(15000, len(X_scaled))
sample_idx = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

k_range = range(2, 11)
inertia_list = []
silhouette_list = []

for k in k_range:
    model = KMeans(n_clusters=k, init='k-means++', n_init=20, max_iter=500, random_state=42)
    labels = model.fit_predict(X_sample)
    inertia_list.append(model.inertia_)
    silhouette_list.append(silhouette_score(X_sample, labels))

best_k = list(k_range)[int(np.argmax(silhouette_list))]
print('Best k by silhouette:', best_k)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(list(k_range), inertia_list, marker='o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[1].plot(list(k_range), silhouette_list, marker='o', color='#D1495B')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=20, max_iter=500, random_state=42)
labels = kmeans.fit_predict(X_scaled)
sil_final = silhouette_score(X_scaled, labels, sample_size=min(10000, len(X_scaled)), random_state=42)

df_clustered = df.loc[df_cluster.index].copy()
df_clustered['KLASTER'] = labels + 1
df_clustered['KLASTER_LABEL'] = df_clustered['KLASTER'].apply(lambda x: f'Klaster {x}')

print(f'Final silhouette score: {sil_final:.4f}')
print(df_clustered['KLASTER'].value_counts().sort_index())

In [ ]:
palette = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(10, 7))
for cluster in sorted(df_clustered['KLASTER'].unique()):
    mask = df_clustered['KLASTER'] == cluster
    ax.scatter(df_clustered.loc[mask, 'IKL_2024'], df_clustered.loc[mask, 'IKE_2024'],
               s=8, alpha=0.3, color=palette[(cluster - 1) % len(palette)], label=f'Klaster {cluster}')
ax.scatter(centroids[:, 2], centroids[:, 1], marker='*', s=300, color='black', label='Centroid')
ax.set_title(f'Clustering Desa berdasarkan IKL dan IKE (k={best_k})')
ax.set_xlabel('IKL 2024')
ax.set_ylabel('IKE 2024')
ax.legend()
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(11, 8))
ax3d = fig.add_subplot(111, projection='3d')
for cluster in sorted(df_clustered['KLASTER'].unique()):
    mask = df_clustered['KLASTER'] == cluster
    ax3d.scatter(df_clustered.loc[mask, 'IKL_2024'], df_clustered.loc[mask, 'IKE_2024'], df_clustered.loc[mask, 'IKS_2024'],
                 s=5, alpha=0.2, color=palette[(cluster - 1) % len(palette)], label=f'Klaster {cluster}')
ax3d.set_title(f'Visualisasi 3D Klaster Desa (k={best_k})')
ax3d.set_xlabel('IKL 2024')
ax3d.set_ylabel('IKE 2024')
ax3d.set_zlabel('IKS 2024')
ax3d.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_clustered.to_csv(CLUSTERED_DATA_PATH, index=False, encoding='utf-8-sig')
print(f'Saved clustered data to {CLUSTERED_DATA_PATH}')